# jia2024

> 对应代码：`EconometricsCode/code_in_notes/Chap.8/jia2024.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


我们读入贾、王、司（2024）使用的微观调查数据，并对政治经济学暴露变量 PEE 取对数，以缓解右偏并便于用弹性解释。

In [2]:
%%stata
clear
use ../datasets/Jia_Wang_Si2024.dta
// 取对数
gen lnPEE=log(PEE)


. clear

. use ../datasets/Jia_Wang_Si2024.dta

. // 取对数
. gen lnPEE=log(PEE)

. 


我们对 lnPEE 做去平均处理，生成离差 dm_lnPEE。

In [3]:
%%stata
// 去平均
egen m_lnPEE=mean(lnPEE)
gen dm_lnPEE=lnPEE-m_lnPEE


. // 去平均
. egen m_lnPEE=mean(lnPEE)

. gen dm_lnPEE=lnPEE-m_lnPEE

. 


我们生成户口虚拟变量与去平均后 lnPEE 的交乘项。使用离差构造交乘项后，主效应 hukou 的系数解释为在 lnPEE 处于样本均值时户口对收入的影响，而不是在 lnPEE=0（通常不在样本支撑范围内）时的影响，解释更有意义。

In [4]:
%%stata
// 交乘项
gen hukou_dm_lnPEE=hukou*dm_lnPEE


. // 交乘项
. gen hukou_dm_lnPEE=hukou*dm_lnPEE
(9 missing values generated)

. 


我们把人口学控制变量存入局部宏，并估计不含任何固定效应的基准 OLS；随后用 outreg2 把结果导出，addt 选项在表格下方标注各组固定效应是否控制。

In [5]:
%%stata
// 回归
local controls "gender nation married parents_edu"
reg lnIncome hukou_dm_lnPEE hukou lnPEE `controls'
outreg2 using jia2024.tex, replace addt(出生年份固定效应, 否, 城市固定效应, 否, 调查年份固定效应, 否, 城市×出生年份固定效应, 否)


. // 回归
. local controls "gender nation married parents_edu"

. reg lnIncome hukou_dm_lnPEE hukou lnPEE `controls'

      Source |       SS           df       MS      Number of obs   =    15,372
-------------+----------------------------------   F(7, 15364)     =    183.03
       Model |  9367.48228         7  1338.21175   Prob > F        =    0.0000
    Residual |  112330.675    15,364  7.31129102   R-squared       =    0.0770
-------------+----------------------------------   Adj R-squared   =    0.0766
       Total |  121698.158    15,371  7.91738713   Root MSE        =    2.7039

------------------------------------------------------------------------------
    lnIncome | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
hukou_dm_l~E |  -.2325481   .0371483    -6.26   0.000    -.3053632    -.159733
       hukou |   .8889249   .0507987    17.50   0.000     .7893535    .9884964
       lnPEE 

我们逐步加入固定效应以控制遗漏变量：先控制出生年份固定效应，吸收同代人共同经历的宏观冲击。

In [6]:
%%stata
reghdfe lnIncome hukou_dm_lnPEE hukou lnPEE `controls', absorb(birthyear)
outreg2 using jia2024.tex, append addt(出生年份固定效应, 是, 城市固定效应, 否, 调查年份固定效应, 否, 城市×出生年份固定效应, 否)


. reghdfe lnIncome hukou_dm_lnPEE hukou lnPEE `controls', absorb(birthyear)
(dropped 6 singleton observations)
(MWFE estimator converged in 1 iterations)

HDFE Linear regression                            Number of obs   =     15,366
Absorbing 1 HDFE group                            F(   7,  15325) =     204.98
                                                  Prob > F        =     0.0000
                                                  R-squared       =     0.0975
                                                  Adj R-squared   =     0.0952
                                                  Within R-sq.    =     0.0856
                                                  Root MSE        =     2.6768

------------------------------------------------------------------------------
    lnIncome | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
hukou_dm_l~E |  -.2884093    .037352    -7.72   0.000

再加入城市和调查年份固定效应，控制城市层面不随时间变化的因素以及所有城市共同的年度冲击。

In [7]:
%%stata
reghdfe lnIncome hukou_dm_lnPEE hukou lnPEE `controls', absorb(birthyear citycode year)
outreg2 using jia2024.tex, append addt(出生年份固定效应, 是, 城市固定效应, 是, 调查年份固定效应, 是, 城市×出生年份固定效应, 否)


. reghdfe lnIncome hukou_dm_lnPEE hukou lnPEE `controls', absorb(birthyear city
> code year)
(dropped 7 singleton observations)
(MWFE estimator converged in 9 iterations)

HDFE Linear regression                            Number of obs   =     15,365
Absorbing 3 HDFE groups                           F(   7,  15153) =     115.51
                                                  Prob > F        =     0.0000
                                                  R-squared       =     0.2388
                                                  Adj R-squared   =     0.2282
                                                  Within R-sq.    =     0.0507
                                                  Root MSE        =     2.4714

------------------------------------------------------------------------------
    lnIncome | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
hukou_dm_l~E |   -.255701   .0377917

最后控制城市×出生年份的交互固定效应（用 i.birthyear#i.citycode 展开为哑变量组），它吸收每个城市每一代人特有的冲击，是最严格的设定；此时单独的出生年份和城市效应被其吸收，故表中标注为“--”。比较各列中交乘项系数的变化，可以判断估计对控制变量选择的稳健性。

In [8]:
%%stata
reghdfe lnIncome hukou_dm_lnPEE hukou lnPEE `controls', absorb(i.birthyear#i.citycode year)
outreg2 using jia2024.tex, append addt(出生年份固定效应, --, 城市固定效应, --, 调查年份固定效应, 是, 城市×出生年份固定效应, 是)


. reghdfe lnIncome hukou_dm_lnPEE hukou lnPEE `controls', absorb(i.birthyear#i.
> citycode year)
(dropped 607 singleton observations)
(MWFE estimator converged in 4 iterations)

HDFE Linear regression                            Number of obs   =     14,765
Absorbing 2 HDFE groups                           F(   7,  13022) =     106.39
                                                  Prob > F        =     0.0000
                                                  R-squared       =     0.3514
                                                  Adj R-squared   =     0.2647
                                                  Within R-sq.    =     0.0541
                                                  Root MSE        =     2.3907

------------------------------------------------------------------------------
    lnIncome | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
hukou_dm_l~E |   -.229886   .0